# PGA DraftKings — weekly (API)

The weekly routine on the PGA Tour's own API. **Nothing to type:** this week's event comes
from the Tour's schedule, last week's results arrive with the refresh, and every golfer —
DraftKings, odds, results — is matched by the Tour's player id.

The features and the model are the same code `pga-dk.ipynb` runs (`utils/features.py`,
`utils/model.py`); only the data underneath differs. Nothing here writes `golf.db`.

Run top to bottom. **At work, skip 4a.** Run **5** before the first tee time.

## 1. Setup

In [ ]:
from IPython import get_ipython

# Autoreload picks up edits to pga_api/ without a restart.
_ip = get_ipython()
if _ip is not None:
    _ip.run_line_magic("load_ext", "autoreload")
    _ip.run_line_magic("autoreload", "2")

import pandas as pd
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

from pga_api import weekly, model
from pga_api.weekly import find_player, add_alias

## 2. Refresh the data

Rebuilds `data/pga.db` from the Tour's API: results of every finished event (last week's
included, automatically), this season's stats, and the entry lists of events starting in
the next ten days.

**Takes seconds:** past results are saved in `data/api_cache/` (committed), so only what can
have changed is downloaded. If that folder is ever missing, the first run takes about five
minutes to fetch everything again.

**Prints** what has finished since the last run. If last week's event is not listed, the
Tour has not marked it final yet — run again later.

In [ ]:
new_events = weekly.refresh()

## 3. This week

The next event to start. When two share the week it takes the one with more of the world's
top 50 in its field — the main event — and names the other. Events whose past editions were
not stroke play (Presidents Cup, Ryder Cup, Zurich) are skipped.

**To use a different event**, pass its id: `weekly.this_week(pick="R2026554")`.
The entry list is usually published the Friday before; until then it reads 0 players,
which is normal.

In [ ]:
week = weekly.this_week()

## 4. DraftKings prices

### 4a. Download · SKIP AT WORK

The only cell that talks to DraftKings (the work network blocks it). Saves the week's file
to `data/salaries/` — **commit it**. Safe to re-run all week; an unchanged file says so.

In [ ]:
from utils import dk_api

dk_api.refresh_from_dk(week.config)

### 4b. Priced field · ALWAYS RUN

Reads the saved file — no network — and matches every DraftKings name to a Tour player id.
**Look for:** `every DraftKings name resolved`. Anything else prints the one line that fixes
it (section 5b).

In [ ]:
dk = weekly.prices(week)
dk.head()

## 5. Odds · BEFORE THE FIRST TEE

The Tour's own FanDuel board, keyed by player id — nothing to match by name, and no way to
pick up another event's board. Saved to `data/odds/` — **commit it**.
The saved board also becomes that event's odds in the training data from then on.

It refuses once the first group has teed off: a forecast built on in-play odds would know
part of round 1. Early in the week the market may not be open yet; it says so, and you run
it again later.

`weekly.odds(week, source="golfodds")` scrapes golfodds.com instead, the old notebook's source.

In [ ]:
odds = weekly.odds(week)

### 5b. Fix a name (only when 4b or 5a asks)

Find the golfer's id, then save the alias. It applies to every source from then on and is
committed in `data/player_aliases.csv`.

In [ ]:
# find_player("Bhatia")
# add_alias("Akshay Bhatia Jr", "56630")

## 6. Report card

Every logged week against its result: how many of the top 15 by `P_TOP20` finished top 20
(the forward test's yardstick is about 6.5), what `P_TOP20` expected, and the top 15's cut
rate. **One row per notebook** — `golf.db notebook` is what `pga-dk.ipynb` logged, `API notebook`
this one — so while both run, the same week shows twice.

In [ ]:
model.report_card()

## 7. Train and score

Trains on every stroke-play event from 2016 to last week (about 62,000 golfer-events,
opposite-field events and TOUR Championships included), each with its features as they stood
the morning it began; then scores this week's priced field. About a minute.

**Look for** odds on nearly the whole field and strokes-gained form on 95%+. Course history
is empty for a course the Tour has not played before — that is normal.

In [ ]:
train, ctx = model.training(as_of=week.end_date)
rows = model.week_rows(ctx, week, dk, odds)
scored, importances = model.train_and_score(train, rows)
scored[["PLAYER", "SALARY", "P_TOP20", "SCORE", "ODDS_SHARE", "SG_FORM"]].head(10)

## 8. Export, log, publish

Saves the scored field (`data/api_week_export.csv`), **logs the forecast** to
`data/predictions/` for the report card — **commit both**, a forecast can only be made before
the event — and writes the dashboard's data. Logging the same week twice changes nothing.

In [ ]:
export_df = model.export(scored, week)
model.log_predictions(export_df, week)
weekly.publish(export_df)
export_df.head(15)

## 9. Open the dashboard

Shows **this notebook's** slate — the top bar reads *API data*. It shares the dashboard with
`pga-dk.ipynb`: whichever notebook opened it last is the one showing. Lineups are saved by
tournament and date, so the same week's lineups are there from either.

Standalone: after a `git pull` on the other computer, this cell alone opens it (seconds).

In [ ]:
from pga_api import weekly

weekly.open_dashboard()

---
## Appendix — checks

Only needed after changing `pga_api/` or while `golf.db` is still in use. Each prints its
reading; none writes.

- `compare.report()` — every difference between `golf.db` and `pga.db`
- `validate.truncation()` — no feature changes when data from the event onward is removed
- `validate.forward_eval()` — both notebooks' out-of-sample record, 2021–2025 (10 minutes)
- `validate.dry_run("R2026557")` — a finished week replayed through sections 4b–7, graded

In [ ]:
from pga_api import compare, validate

# r = compare.report()
# validate.truncation()
# validate.dry_run("R2026557")